In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "lonboard[xarray]>0.17.0",
#     "numpy>=2",
#     "obstore>=0.9.2",
#     "palettable>=3.3.3",
#     "pillow>=12.1.1",
#     "xarray>=2025.1",
#     "zarr>=3",
# ]
# ///

# Visualize multiscale GeoZarr with xarray in Lonboard

Lonboard can render any [xarray](https://xarray.dev/) `DataArray` on a regular grid, and a multiscale pyramid opened as a `DataTree`. This notebook shows a 10 m digital elevation model of the contiguous United States, stored as a [GeoZarr](https://geozarr.org/) pyramid with fourteen resolution levels.

The data is read in Python, tile by tile, as you pan and zoom. Zoomed-out views read the coarse levels of the pyramid, so the whole country costs a handful of small reads, and zooming in reads only the chunks under the view at a finer level.

The dataset is published by [carbonplan](https://carbonplan.org/) as one of the examples for their [zarr-layer](https://github.com/carbonplan/zarr-layer) project.

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run examples/raster-xarray.ipynb
```

(The `uvx` command is included when installing `uv`).

## Imports

We use [Obstore](https://developmentseed.org/obstore/latest/) to read the store over HTTP, [Zarr](https://zarr.dev/) and [xarray](https://xarray.dev/) to open it, and [pillow](https://pillow.readthedocs.io/en/stable/) (imported as `PIL`) to encode image tiles as PNG.

In [ ]:
import io

import numpy as np
import xarray as xr
from obstore.store import HTTPStore
from palettable.scientific.sequential import Batlow_20
from PIL import Image
from zarr.storage import ObjectStore

from lonboard import Map, RasterLayer
from lonboard.colormap import apply_continuous_cmap
from lonboard.raster import EncodedImage

## Open the pyramid as a `DataTree`

The store has consolidated metadata, so opening it is a single request. Each child group of the tree is one resolution level, and the root attributes describe the pyramid with the [zarr-conventions multiscales](https://github.com/zarr-conventions/multiscales) layout.

In [ ]:
url = "https://carbonplan-share.s3.us-west-2.amazonaws.com/zarr-layer-examples/USGS-CONUS-DEM-10m.zarr"
store = ObjectStore(HTTPStore(url), read_only=True)
tree = xr.open_datatree(
    store,
    engine="zarr",
    chunks=None,
    zarr_format=3,
    consolidated=True,
)
print("levels:", list(tree.children))
print("variables:", list(tree["0"].data_vars))

## Create the render callback

Lonboard calls this function with one tile at a time: a `DataArray` window of the level being drawn, with its coordinates. Here we stretch elevation from 0 to 3500 metres through a colormap, and make `NaN` (the ocean and everything outside the United States) transparent.

In [ ]:
def render_dem(tile: xr.DataArray) -> EncodedImage:
    """Color elevation in metres and make NaN transparent."""
    values = tile.values
    mask = np.isnan(values)
    normalized = np.clip(np.nan_to_num(values) / 3500, 0, 1)
    rgb = apply_continuous_cmap(normalized.ravel(), Batlow_20).reshape(*values.shape, 3)
    alpha = np.where(mask, 0, 255).astype(np.uint8)
    rgba = np.concatenate([rgb, alpha[..., None]], axis=-1)

    img = Image.fromarray(rgba)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

## Create the layer

Pass the tree and the callback to `RasterLayer.from_xarray`. Each level group holds several variables, so we name the one to draw. Lonboard reads the grid from each level's coordinates, sizes its tiles from the store's chunks, and sets `min_zoom` so that a zoomed-out view never requests more tiles than the coarsest level has.

In [ ]:
layer = RasterLayer.from_xarray(tree, render_tile=render_dem, variable="DEM")

Now create a map and display it like any other layer.

In [ ]:
m = Map(layer, height=600)
m

## Debugging the render callback

The coarsest level fits in one tile, so we can run the callback on it directly to check the image it produces.

In [ ]:
coarsest = list(tree.children)[-1]
render_dem(tree[coarsest].to_dataset()["DEM"].load())